In [ ]:
# ============================================================
# جلد کی بیماریاں - EfficientNet-B4 مکمل Training Code
# Dataset: E:\Skin DataSet\Train اور E:\Skin DataSet\Val
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize
from pathlib import Path
import cv2
from tqdm import tqdm
import os

# ============================================================
# 1. Dataset Paths - آپ کے راستے
# ============================================================
TRAIN_DIR = r"E:\Skin DataSet\Train"
VAL_DIR   = r"E:\Skin DataSet\Val"

# ============================================================
# 2. کلاسز کی لسٹ (10 بیماریاں)
# ============================================================
CLASSES = [
    'Eczema',
    'Melanoma',
    'Atopic Dermatitis',
    'Basal Cell Carcinoma (BCC)',
    'Melanocytic Nevi (NV)',
    'Benign Keratosis-like Lesions (BKL)',
    'Psoriasis Lichen Planus',
    'Seborrheic Keratoses',
    'Tinea Ringworm Candidiasis',
    'Warts Molluscum'
]
NUM_CLASSES = len(CLASSES)

# Image size EfficientNet-B4 کے لیے
IMG_SIZE    = 380
BATCH_SIZE  = 16   # اگر GPU memory کم ہو تو 8 کریں
EPOCHS      = 30
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# ============================================================
# 3. Augmentation Pipelines
# ============================================================
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3),
    A.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1,
        p=0.5
    ),
    A.GaussNoise(p=0.2),
    A.OneOf([
        A.ElasticTransform(p=0.5),
        A.GridDistortion(p=0.5),
    ], p=0.3),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
    ToTensorV2()
])

# ============================================================
# 4. Dataset Class - فولڈر سے خودبخود لوڈ ہوگا
# ============================================================
class SkinDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir  = Path(root_dir)
        self.transform = transform
        self.image_paths = []
        self.labels      = []

        # فولڈر کا نام class نام سے match کریں
        self.class_to_idx = {}
        found_folders = sorted([
            d.name for d in self.root_dir.iterdir() if d.is_dir()
        ])

        print(f"\nملی ہوئی کلاسز ({root_dir}):")
        for idx, folder_name in enumerate(found_folders):
            self.class_to_idx[folder_name] = idx
            print(f"  {idx}: {folder_name}")

            folder_path = self.root_dir / folder_name
            for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
                for img_path in folder_path.glob(ext):
                    self.image_paths.append(img_path)
                    self.labels.append(idx)

        print(f"کل تصاویر: {len(self.image_paths)}\n")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image    = cv2.imread(str(img_path))

        # اگر تصویر نہ ملے
        if image is None:
            image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            image = self.transform(image=image)['image']

        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label

# ============================================================
# 5. CutMix Augmentation
# ============================================================
def cutmix(images, labels, alpha=1.0):
    batch_size = images.size(0)
    lam        = np.random.beta(alpha, alpha)

    rand_index = torch.randperm(batch_size).to(DEVICE)
    labels_a   = labels
    labels_b   = labels[rand_index]

    W, H   = images.size(2), images.size(3)
    cut_w  = int(W * np.sqrt(1 - lam))
    cut_h  = int(H * np.sqrt(1 - lam))

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    x1 = max(cx - cut_w // 2, 0)
    x2 = min(cx + cut_w // 2, W)
    y1 = max(cy - cut_h // 2, 0)
    y2 = min(cy + cut_h // 2, H)

    images[:, :, x1:x2, y1:y2] = images[rand_index, :, x1:x2, y1:y2]
    lam = 1 - ((x2 - x1) * (y2 - y1) / (W * H))

    return images, labels_a, labels_b, lam

# ============================================================
# 6. EfficientNet-B4 Model
# ============================================================
class SkinModel(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b4',
            pretrained=True,
            num_classes=0  # صرف features
        )
        self.dropout    = nn.Dropout(0.4)
        self.classifier = nn.Linear(
            self.backbone.num_features,
            num_classes
        )

    def forward(self, x):
        features = self.backbone(x)
        features = self.dropout(features)
        return self.classifier(features)

# ============================================================
# 7. Class Imbalance Handle - WeightedSampler
# ============================================================
def get_sampler(dataset):
    labels       = np.array(dataset.labels)
    class_counts = np.bincount(labels)
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[labels]
    sampler = WeightedRandomSampler(
        weights     = torch.FloatTensor(sample_weights),
        num_samples = len(sample_weights),
        replacement = True
    )
    return sampler

# ============================================================
# 8. Training Function
# ============================================================
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct    = 0
    total      = 0

    for images, labels in tqdm(loader, desc="Training"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        # 50% chance CutMix لگائیں
        if np.random.random() > 0.5:
            images, labels_a, labels_b, lam = cutmix(images, labels)
            outputs = model(images)
            loss    = (lam * criterion(outputs, labels_a) +
                       (1 - lam) * criterion(outputs, labels_b))
        else:
            outputs = model(images)
            loss    = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy

# ============================================================
# 9. Validation Function
# ============================================================
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validation"):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss    = criterion(outputs, labels)

            total_loss += loss.item()
            probs = torch.softmax(outputs, dim=1)
            all_preds.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    # AUC-ROC
    all_labels_bin = label_binarize(all_labels, classes=range(NUM_CLASSES))
    auc = roc_auc_score(
        all_labels_bin, all_preds,
        average='macro', multi_class='ovr'
    )

    accuracy = (all_preds.argmax(1) == all_labels).mean() * 100
    avg_loss = total_loss / len(loader)
    return avg_loss, accuracy, auc

# ============================================================
# 10. Main - سب کچھ یہاں چلتا ہے
# ============================================================
def main():
    print("=" * 55)
    print("جلد کی بیماریاں - EfficientNet-B4 Training")
    print("=" * 55)

    # Dataset لوڈ کریں
    train_dataset = SkinDataset(TRAIN_DIR, transform=train_transform)
    val_dataset   = SkinDataset(VAL_DIR,   transform=val_transform)

    # Sampler - class imbalance ٹھیک کریں
    sampler = get_sampler(train_dataset)

    # DataLoader بنائیں
    train_loader = DataLoader(
        train_dataset,
        batch_size  = BATCH_SIZE,
        sampler     = sampler,
        num_workers = 4,
        pin_memory  = True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size  = BATCH_SIZE,
        shuffle     = False,
        num_workers = 4,
        pin_memory  = True
    )

    # Model بنائیں
    model = SkinModel(num_classes=NUM_CLASSES).to(DEVICE)

    # Loss - label smoothing سے بہتر نتائج
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # Phase 1: صرف classifier train کریں (5 epochs)
    optimizer = optim.AdamW(
        model.classifier.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS
    )

    best_auc    = 0
    best_epoch  = 0

    print(f"\nTraining شروع - کل {EPOCHS} Epochs")
    print("-" * 55)

    for epoch in range(EPOCHS):

        # Phase 2: epoch 5 پر پورا backbone unfreeze
        if epoch == 5:
            print("\nBackbone unfreeze کر رہے ہیں - Fine-tuning شروع")
            optimizer = optim.AdamW([
                {'params': model.backbone.parameters(), 'lr': 1e-4},
                {'params': model.classifier.parameters(), 'lr': 1e-3}
            ], weight_decay=1e-4)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=EPOCHS - 5
            )

        print(f"\nEpoch [{epoch + 1}/{EPOCHS}]")

        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion
        )
        val_loss, val_acc, val_auc = validate(
            model, val_loader, criterion
        )

        scheduler.step()

        print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"  Val   Loss: {val_loss:.4f}  | Val   Acc: {val_acc:.2f}%  | AUC: {val_auc:.4f}")

        # بہترین model محفوظ کریں
        if val_auc > best_auc:
            best_auc   = val_auc
            best_epoch = epoch + 1
            torch.save({
                'epoch'      : epoch,
                'model_state': model.state_dict(),
                'optimizer'  : optimizer.state_dict(),
                'auc'        : val_auc,
                'accuracy'   : val_acc,
            }, 'best_skin_model.pth')
            print(f"  ✓ بہترین Model محفوظ! AUC: {val_auc:.4f}")

    print("\n" + "=" * 55)
    print(f"Training مکمل!")
    print(f"بہترین AUC: {best_auc:.4f} - Epoch {best_epoch}")
    print("Model: best_skin_model.pth میں محفوظ ہے")
    print("=" * 55)

# ============================================================
# چلائیں
# ============================================================
if __name__ == "__main__":
    main()